In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

In [5]:
url = 'https://web.archive.org/web/2021/https://www.ccny.cuny.edu/registrar/fall'
r = requests.get(url)
print(r.status_code)

soup = BeautifulSoup(r.text, 'html.parser')
table = soup.find('table')
rows = table.find_all('tr')
len(rows)

200


37

In [6]:
def parse_date(s):
    # "August 25 - 31" -> use the start date "August 25"
    # "January 1, 2022" -> has its own year
    m = re.match(r'([A-Za-z]+)\s+(\d+)(?:,\s*(\d{4}))?', s)
    if m is None:
        return None  # header row
    month, day, year = m.groups()
    return datetime.strptime(f"{month} {day} {year or 2021}", "%B %d %Y").date()

records = []
for row in rows:
    cells = row.find_all(['td', 'th'])
    if len(cells) < 3:
        continue
    date = parse_date(cells[0].get_text(strip=True))
    if date is None:
        continue
    records.append({
        'date': date,
        'dow': cells[1].get_text(strip=True),
        'text': ' '.join(cells[2].get_text().split()).rstrip(';'),
    })

len(records)

36

In [8]:
df = pd.DataFrame(records).set_index('date')
df['dow'] = df['dow'].str.split(' - ').str[0]  # ranges like "Wednesday - Tuesday" -> start day
df

,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-28,Saturday,First day of Saturday Classes
2021-08-31,Tuesday,Last day to add a class to an existing enrollm...
2021-09-01,Wednesday,Verification of Enrollment rosters available t...
